# Lab 09: Multiple Linear Regression
> Week 9 | CLO3 | ISLP Ch.3.2

## บทนำสัปดาห์

สัปดาห์นี้เราจะขยาย Simple Linear Regression ให้รองรับ predictor ได้หลายตัวพร้อมกันผ่าน **Multiple Linear Regression (MLR)** ซึ่งเป็น model ที่ใช้จริงมากที่สุดใน Data Science เพราะปัญหาจริงมักมีตัวแปรที่เกี่ยวข้องมากกว่าหนึ่งตัว ความแตกต่างสำคัญจาก SLR คือ coefficient β̂ⱼ ใน MLR หมายถึงผลของ Xⱼ **โดยยึด predictor อื่นคงที่** ซึ่งช่วยแก้ปัญหา confounding ที่เกิดจาก correlation ระหว่าง predictors ใน lab นี้คุณจะสร้าง MLR ด้วย Normal Equations (Week 3), sklearn และ statsmodels ทดสอบ F-statistic ทำ variable selection ด้วย Adjusted R² และ AIC และสุดท้ายสร้าง CI/PI สำหรับ prediction ใหม่ ทักษะเหล่านี้ใช้ในทุก domain ตั้งแต่การวิเคราะห์ยอดขายไปจนถึงการพยากรณ์ราคาบ้าน

**LLo**: สร้าง MLR, ตีความ F-statistic และเลือก predictor ที่สำคัญได้

**สิ่งที่จะเรียนรู้**:
- Part 1: MLR model + Normal Equations connection (Week 3)
- Part 2: F-statistic และ t-statistic — ความต่าง
- Part 3: Variable selection — Adjusted R², AIC, BIC
- Part 4: CI vs PI สำหรับ prediction
- Part 5: Case Study — เปรียบเทียบ SLR vs MLR


In [ ]:
# ─── ติดตั้ง ISLP package ───────────────────────────────────────────
# วัตถุประสงค์: ติดตั้งไลบรารี ISLP ซึ่งเป็นแหล่งข้อมูลทางการของหนังสือ
# (James, Witten, Hastie, Tibshirani, Taylor) ที่รวม dataset ส่วนใหญ่ในเล่มไว้ให้โหลดตรง ๆ
# อ้างอิง: statlearning.com -> Python package "ISLP"
!pip install -q ISLP
print('ติดตั้ง ISLP เสร็จแล้ว')

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ที่ใช้ในทั้ง lab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from itertools import combinations

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
np.random.seed(42)
print('Libraries loaded ✓')

In [ ]:
# ─── โหลด Advertising dataset ──────────────────────────────────────
# วัตถุประสงค์: ใช้ข้อมูลงบโฆษณา 200 ตลาดเพื่อสร้าง MLR model
# ลำดับการหาแหล่งข้อมูล: 1) ISLP package  2) URL ของ statlearning.com  3) synthetic data จำลอง
# หมายเหตุ: Advertising ไม่ได้ฝังอยู่ใน ISLP package (เป็นตัวอย่างเปิดเรื่องใน Ch.2 ไม่ใช่ Lab dataset
# ใน Table 1.1) จึงคาดว่าขั้นตอน ISLP จะไม่พบและตกไปใช้ URL ตามที่หนังสือสอนจริง แต่ยังลองก่อน
# เผื่อเวอร์ชันแพ็กเกจในอนาคตเพิ่มเข้ามา
try:
    from ISLP import load_data
    df = load_data('Advertising')
    print('Loaded via ISLP package ✓')
except Exception:
    try:
        df = pd.read_csv('https://www.statlearning.com/s/Advertising.csv', index_col=0)
        print('Loaded via statlearning.com URL ✓')
    except Exception:
        # Fallback: synthetic data ที่มีลักษณะเดียวกัน
        np.random.seed(0)
        n = 200
        TV       = np.random.uniform(0.7, 296.4, n)
        Radio    = np.random.uniform(0, 49.6, n)
        News     = np.random.uniform(0.3, 114, n)
        Sales    = 2.94 + 0.046*TV + 0.189*Radio - 0.001*News + np.random.normal(0, 1.69, n)
        df = pd.DataFrame({'TV': TV, 'Radio': Radio, 'Newspaper': News, 'Sales': Sales})
        print('Using synthetic data (Advertising.csv not found)')

print(f'Shape: {df.shape}')
df.head()

---
## Part 1: MLR Model และ Normal Equations (Week 3 Connection)

**Part นี้เราจะสร้าง MLR** เพื่อให้เห็นว่า Multiple Linear Regression ไม่ใช่สิ่งใหม่ — มันคือ Normal Equations (XᵀX)β̂ = Xᵀy ที่เรียนใน Week 3 นำมาใช้กับ p > 1 predictors

เราจะ implement ด้วย 3 วิธีและ verify ว่าให้ผลเท่ากัน:
1. `np.linalg.lstsq()` — Normal Equations โดยตรง
2. `sklearn.LinearRegression` — เร็วสำหรับ prediction
3. `statsmodels.ols()` — ได้ full inference (SE, t, p, F, CI)


In [ ]:
# ─── วิธีที่ 1: Normal Equations ──────────────────────────────────
# วัตถุประสงค์: สร้าง design matrix X แล้วใช้ β̂ = (XᵀX)⁻¹Xᵀy จาก Week 3

# สร้าง design matrix: column แรกเป็น 1 สำหรับ intercept
X_mat = np.column_stack([
    np.ones(len(df)),   # intercept column
    df['TV'].values,
    df['Radio'].values,
    df['Newspaper'].values
])
y_vec = df['Sales'].values

print(f'Design matrix X shape: {X_mat.shape}')  # 200×4
print(f'y shape: {y_vec.shape}')                 # 200

# แก้ Normal Equations ด้วย lstsq (stable กว่า inverse โดยตรง)
beta_lstsq, _, _, _ = np.linalg.lstsq(X_mat, y_vec, rcond=None)
print('\nNormal Equations β̂:')
labels = ['Intercept', 'TV', 'Radio', 'Newspaper']
for lbl, val in zip(labels, beta_lstsq):
    print(f'  {lbl:12s}: {val:.4f}')

In [ ]:
# ─── วิธีที่ 2: sklearn ────────────────────────────────────────────
# วัตถุประสงค์: สะดวกสำหรับ prediction pipeline
sk_model = LinearRegression().fit(df[['TV','Radio','Newspaper']], df['Sales'])
print('sklearn β̂:')
print(f'  Intercept: {sk_model.intercept_:.4f}')
for name, coef in zip(['TV','Radio','Newspaper'], sk_model.coef_):
    print(f'  {name:12s}: {coef:.4f}')

# ─── วิธีที่ 3: statsmodels ─────────────────────────────────────────
# วัตถุประสงค์: ได้ full inference ครบถ้วน
sm_model = smf.ols('Sales ~ TV + Radio + Newspaper', data=df).fit()
print('\nstatsmodels β̂:')
print(sm_model.params.round(4))

# ─── verify ว่าทั้ง 3 วิธีให้ผลเท่ากัน ─────────────────────────────
# วัตถุประสงค์: แสดงว่า implementation ต่างกันแต่ math เดียวกัน
sk_betas = np.concatenate([[sk_model.intercept_], sk_model.coef_])
sm_betas = sm_model.params.values
print('\nVerify allclose (Normal Eq vs sklearn):', np.allclose(beta_lstsq, sk_betas, atol=1e-6))
print('Verify allclose (Normal Eq vs statsmodels):', np.allclose(beta_lstsq, sm_betas, atol=1e-6))

In [ ]:
# ─── แสดง full statsmodels summary ────────────────────────────────
# วัตถุประสงค์: อ่าน F-stat, t-stat, p-value, CI, R², R²_adj, AIC
print(sm_model.summary())

In [ ]:
# ─── เปรียบเทียบ SLR vs MLR coefficients ───────────────────────────
# วัตถุประสงค์: แสดงว่า confounding ทำให้ coefficient เปลี่ยนเมื่อเพิ่ม predictor
slr_tv     = smf.ols('Sales ~ TV', data=df).fit()
slr_radio  = smf.ols('Sales ~ Radio', data=df).fit()
slr_news   = smf.ols('Sales ~ Newspaper', data=df).fit()

print('Coefficient comparison: SLR vs MLR')
print(f'{"Model":30s} {"β̂_TV":>10} {"β̂_Radio":>10} {"β̂_News":>12} {"R²":>8}')
print('-'*72)
print(f'{"SLR(TV)": <30} {slr_tv.params["TV"]:>10.4f} {"—":>10} {"—":>12} {slr_tv.rsquared:>8.4f}')
print(f'{"SLR(Radio)":30} {"—":>10} {slr_radio.params["Radio"]:>10.4f} {"—":>12} {slr_radio.rsquared:>8.4f}')
print(f'{"SLR(Newspaper)":30} {"—":>10} {"—":>10} {slr_news.params["Newspaper"]:>12.4f} {slr_news.rsquared:>8.4f}')
print(f'{"MLR(TV+Radio+News)":30} {sm_model.params["TV"]:>10.4f} {sm_model.params["Radio"]:>10.4f} {sm_model.params["Newspaper"]:>12.4f} {sm_model.rsquared:>8.4f}')

### TODO 1 (Easy): เปรียบเทียบ β̂ และ ตีความ confounding

จากตาราง coefficient comparison ที่เพิ่งสร้าง ให้สังเกตว่า β̂_Newspaper เปลี่ยนจาก 0.055 (SLR) เป็น −0.001 (MLR)

**สิ่งที่ต้องทำ**:
1. คำนวณ correlation matrix ระหว่าง TV, Radio, Newspaper
2. ดู correlation ระหว่าง Newspaper และ Radio
3. อธิบายใน markdown cell ว่าทำไม β̂_Newspaper เปลี่ยนมากเมื่อเพิ่ม Radio เข้าไป


In [ ]:
# TODO 1: คำนวณ correlation matrix
# วัตถุประสงค์: แสดงว่า predictors correlated กันอย่างไร — ทำให้ β̂ เปลี่ยนเมื่อ add predictor

# เติม code ที่นี่
# 1. df[['TV','Radio','Newspaper','Sales']].corr()
# 2. plot heatmap
# 3. ระบุ correlation ระหว่าง Newspaper-Radio และ Newspaper-TV

---
## Part 2: F-statistic และ t-statistic

**Part นี้เราจะทำความเข้าใจ F-statistic** ซึ่งทดสอบว่า predictor **ทั้งหมด** มีความสัมพันธ์กับ Y หรือไม่ (H₀: β₁=β₂=…=βₚ=0) และเปรียบเทียบกับ t-statistic ที่ทดสอบ predictor **ทีละตัว**

ความแตกต่างสำคัญ:
- **F-test**: "model มีประโยชน์โดยรวมไหม?" — ดูก่อนเสมอ
- **t-test**: "predictor ตัวนี้จำเป็นไหม?" — ดูหลังจาก F significant


In [ ]:
# ─── อ่าน F-statistic จาก summary ─────────────────────────────────
# วัตถุประสงค์: อ่านค่า F และ p-value ก่อน แล้วค่อย verify ด้วยสูตร
print(f'F-statistic: {sm_model.fvalue:.2f}')
print(f'F p-value:   {sm_model.f_pvalue:.2e}')
print(f'Interpretation: F={sm_model.fvalue:.2f}, p={sm_model.f_pvalue:.2e} < 0.05')
print('→ Reject H₀: β_TV = β_Radio = β_Newspaper = 0')
print('→ At least one predictor is related to Sales')

In [ ]:
# ─── verify F-statistic ด้วยสูตร ──────────────────────────────────
# วัตถุประสงค์: เข้าใจว่า F = MSR/MSE มาจากไหน
y     = df['Sales'].values
y_hat = sm_model.fittedvalues.values
y_bar = y.mean()
n, p  = len(y), 3  # 3 predictors

TSS = ((y - y_bar)**2).sum()
RSS = ((y - y_hat)**2).sum()
MSR = (TSS - RSS) / p
MSE = RSS / (n - p - 1)
F_manual = MSR / MSE

print(f'TSS = {TSS:.2f}')
print(f'RSS = {RSS:.2f}')
print(f'MSR = (TSS-RSS)/p = {MSR:.2f}')
print(f'MSE = RSS/(n-p-1) = {MSE:.4f}')
print(f'F   = MSR/MSE     = {F_manual:.2f}')
print(f'Verify: model.fvalue = {sm_model.fvalue:.2f} ✓')

### TODO 2 (Medium): เปรียบเทียบ F-test กับ t-test

เราเห็นว่า F significant (p ≈ 0) และ Newspaper t NOT significant (p = 0.86)

**สิ่งที่ต้องทำ**:
1. สร้างตาราง DataFrame ที่แสดง coef, SE, t, p-value, significant? ของทุก predictor
2. ตรวจสอบ: ใน SLR, F = t² — verify บน Sales ~ TV
3. อธิบายใน markdown cell ว่า:
   - ทำไม F significant แต่ Newspaper t NOT significant ไม่ขัดแย้งกัน
   - ทำไมเราต้อง check F-test ก่อน t-test


In [ ]:
# TODO 2: เปรียบเทียบ F-test กับ t-test
# วัตถุประสงค์: เห็นความสัมพันธ์และความต่างระหว่าง F-test (overall) และ t-test (individual)

# เติม code ที่นี่
# 1. สร้าง DataFrame ของ coef, SE, t, p-value
# 2. verify F = t² สำหรับ SLR
# 3. plot t-values เป็น bar chart พร้อมเส้น critical value ±2

---
## Part 3: Variable Selection

**Part นี้เราจะหา best model** โดยลอง subset ของ predictors ทั้งหมดและเปรียบเทียบด้วย Adjusted R² และ AIC

เหตุผลที่ต้องใช้ R²_adj แทน R² คือ R² ไม่เคยลดเมื่อเพิ่ม predictor แม้ predictor นั้นไม่มีประโยชน์ ส่วน Adjusted R² จะลดถ้า predictor ใหม่ไม่ช่วยอธิบาย Y มากพอที่จะคุ้มกับ complexity ที่เพิ่มขึ้น


In [ ]:
# ─── Best Subset Selection ─────────────────────────────────────────
# วัตถุประสงค์: ลอง model ทุก subset เพื่อหา combination ที่ดีที่สุด
predictors = ['TV', 'Radio', 'Newspaper']
results = []

for r in range(1, len(predictors) + 1):
    for combo in combinations(predictors, r):
        formula = 'Sales ~ ' + ' + '.join(combo)
        m = smf.ols(formula, data=df).fit()
        results.append({
            'model': ' + '.join(combo),
            'p': len(combo),
            'R2':     round(m.rsquared, 4),
            'R2_adj': round(m.rsquared_adj, 4),
            'AIC':    round(m.aic, 1),
            'BIC':    round(m.bic, 1),
            'RSE':    round(np.sqrt(m.mse_resid), 4)
        })

res_df = pd.DataFrame(results).sort_values('AIC')
print('All subsets sorted by AIC:')
print(res_df.to_string(index=False))

In [ ]:
# ─── Visualize model comparison ────────────────────────────────────
# วัตถุประสงค์: เห็นภาพว่า TV+Radio เป็น best model อย่างชัดเจน
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AIC bar chart
axes[0].barh(res_df['model'], res_df['AIC'], color='steelblue', alpha=0.8)
axes[0].set_xlabel('AIC (lower = better)')
axes[0].set_title('Model Comparison: AIC')
axes[0].axvline(res_df['AIC'].min(), color='red', lw=1.5, linestyle='--', label='Best')
axes[0].legend()

# R²_adj bar chart
axes[1].barh(res_df['model'], res_df['R2_adj'], color='coral', alpha=0.8)
axes[1].set_xlabel('Adjusted R² (higher = better)')
axes[1].set_title('Model Comparison: Adjusted R²')

plt.tight_layout()
plt.show()

### TODO 3 (Medium): Backward Elimination

ทำ backward elimination ด้วย threshold p = 0.05 เริ่มจาก full model (TV+Radio+Newspaper)

**สิ่งที่ต้องทำ**:
1. Fit full model
2. หา predictor ที่มี p-value สูงที่สุด — ถ้า > 0.05 → ลบ
3. Fit model ใหม่ ทำซ้ำจนทุก predictor significant
4. พิมพ์ผลแต่ละ step และสรุป final model


In [ ]:
# TODO 3: Backward Elimination
# วัตถุประสงค์: ทำ backward selection step-by-step เพื่อเห็นกระบวนการ

# เติม code ที่นี่
# Hint: 
#   current = ['TV', 'Radio', 'Newspaper']
#   while True:
#       fit model
#       find max p-value
#       if max_p > 0.05: remove that predictor
#       else: break

---
## Part 4: Confidence Intervals vs Prediction Intervals

**Part นี้เราจะสร้าง CI และ PI** เพื่อเห็นความต่างระหว่าง "predict mean response" กับ "predict individual response"

- **CI**: uncertainty ของ f(X₀) = β₀ + β₁X₁ + β₂X₂ — ตอบคำถาม "ค่าเฉลี่ย Sales เมื่อ TV=x₀ คือเท่าไร?"
- **PI**: uncertainty ของ ŷ + ε — ตอบคำถาม "ตลาดนี้จะขายได้เท่าไร?" (PI กว้างกว่าเสมอ)


In [ ]:
# ─── Fit best model (TV + Radio) ──────────────────────────────────
# วัตถุประสงค์: ใช้ best model จาก Part 3 เพื่อสร้าง prediction intervals
best_model = smf.ols('Sales ~ TV + Radio', data=df).fit()
print(best_model.summary())

In [ ]:
# ─── CI และ PI สำหรับ point เฉพาะ ─────────────────────────────────
# วัตถุประสงค์: ดูค่า CI และ PI สำหรับ TV=100, Radio=20
new_obs = pd.DataFrame({'TV': [100], 'Radio': [20]})
pred    = best_model.get_prediction(new_obs)
frame   = pred.summary_frame(alpha=0.05)

print('Prediction for TV=100, Radio=20:')
print(frame[['mean','mean_ci_lower','mean_ci_upper','obs_ci_lower','obs_ci_upper']].round(2))
print(f"\nŷ  = {frame['mean'].values[0]:.2f} พันหน่วย")
print(f"CI = ({frame['mean_ci_lower'].values[0]:.2f}, {frame['mean_ci_upper'].values[0]:.2f}) — mean response")
print(f"PI = ({frame['obs_ci_lower'].values[0]:.2f}, {frame['obs_ci_upper'].values[0]:.2f}) — individual")

### TODO 4 (Hard): CI/PI Plot บน TV range

สร้าง visualization แสดง fitted line, CI band และ PI band สำหรับ Sales ~ TV (ยึด Radio=20 คงที่)

**สิ่งที่ต้องทำ**:
1. สร้าง grid ของ TV values จาก min ถึง max (100 จุด) ยึด Radio=20
2. คำนวณ CI และ PI สำหรับแต่ละจุด
3. Plot: scatter + fitted line + CI band + PI band
4. ตีความ: อธิบายว่า CI และ PI ต่างกันอย่างไรและใช้เมื่อไร


In [ ]:
# TODO 4: CI/PI visualization
# วัตถุประสงค์: เห็นภาพชัดว่า PI กว้างกว่า CI — ช่วยตัดสินใจว่าจะใช้ตัวใดในสถานการณ์จริง

# เติม code ที่นี่
# Hint:
#   tv_grid = pd.DataFrame({'TV': np.linspace(min, max, 100), 'Radio': 20})
#   pred = best_model.get_prediction(tv_grid)
#   frame = pred.summary_frame(alpha=0.05)
#   plt.fill_between(tv_grid['TV'], frame['mean_ci_lower'], frame['mean_ci_upper'], ...)
#   plt.fill_between(tv_grid['TV'], frame['obs_ci_lower'], frame['obs_ci_upper'], ...)

---
## Part 5: Case Study — ASEAN E-Commerce Sales Prediction

**Scenario**: บริษัท E-Commerce ใน ASEAN ต้องการ predict ยอดขาย (Sales, พัน USD) จากงบโฆษณา 3 ช่องทาง: Social Media, Search Engine Ads, และ Email Marketing

**Data**: synthetic แต่สมจริง — 150 campaigns


In [ ]:
# ─── Case Study: E-Commerce Sales ─────────────────────────────────
# วัตถุประสงค์: ใช้ MLR กับ dataset ใหม่เพื่อให้เห็น workflow จริง
np.random.seed(123)
n_cs = 150

# สร้าง data: Social Media และ Search Ads มีผลต่อ Sales; Email มีน้อย
Social = np.random.uniform(5, 100, n_cs)   # งบ Social Media (พัน USD)
Search = np.random.uniform(2, 80, n_cs)    # งบ Search Ads (พัน USD)
Email  = np.random.uniform(1, 30, n_cs)    # งบ Email (พัน USD)
# true relationship: Social: 0.8, Search: 1.2, Email: 0.1
Sales_ecomm = (5 + 0.8*Social + 1.2*Search + 0.1*Email
               + np.random.normal(0, 8, n_cs))

df_ecomm = pd.DataFrame({
    'Social': Social, 'Search': Search,
    'Email': Email, 'Sales': Sales_ecomm
})

print('E-Commerce dataset:')
print(df_ecomm.describe().round(2))

### TODO 5 (Hard): Full MLR Analysis บน E-Commerce Data

ทำ complete MLR analysis บน E-Commerce dataset และเขียน business report

**สิ่งที่ต้องทำ**:
1. Fit MLR: Sales ~ Social + Search + Email ด้วย statsmodels
2. ตรวจ F-statistic: model significant ไหม?
3. ดู t-statistic ของแต่ละตัว: ตัวใด significant?
4. ทำ best subset selection เปรียบ R²_adj และ AIC
5. Fit best model และ visualize: actual vs predicted scatter plot + residual plot
6. Predict Sales สำหรับ campaign ใหม่: Social=50, Search=40, Email=10
   - รายงาน ŷ, 95% CI, 95% PI
7. เขียน Business Report 5–7 ประโยค สำหรับ Marketing Director


In [ ]:
# TODO 5: Full MLR Analysis บน E-Commerce
# วัตถุประสงค์: ฝึก end-to-end workflow ของ regression analysis จริง

# เติม code ที่นี่

---
## สรุป Lab 09

| Concept | สูตร / Method | Python |
|---------|-------------|--------|
| MLR model | Y = Xβ + ε | `smf.ols('y ~ x1+x2', data).fit()` |
| Normal Equations | β̂ = (XᵀX)⁻¹Xᵀy | `np.linalg.lstsq(X, y)` |
| F-statistic | MSR/MSE = (TSS−RSS)/p ÷ RSS/(n−p−1) | `model.fvalue`, `model.f_pvalue` |
| Adjusted R² | 1 − RSS/(n−p−1) / TSS/(n−1) | `model.rsquared_adj` |
| AIC/BIC | information criteria | `model.aic`, `model.bic` |
| CI | ŷ ± t*·SE_mean | `model.get_prediction().summary_frame()['mean_ci_*']` |
| PI | ŷ ± t*·SE_pred | `model.get_prediction().summary_frame()['obs_ci_*']` |

## Reflection Questions

1. **Confounding**: ทำไม β̂_Newspaper จาก SLR (significant) กับ MLR (not significant) ต่างกันมาก? สิ่งนี้บอกอะไรเกี่ยวกับการตีความ regression coefficient?

2. **F vs t**: ในกรณีที่ F-statistic significant แต่ t-statistic ของทุก predictor ไม่ significant เกิดขึ้นได้ไหม? ถ้าเกิด แปลว่าอะไร?

3. **CI vs PI**: ถ้า Advertising ต้องการรู้ว่า "ตลาดใหม่ที่ใช้ TV=$200k จะขายได้เท่าไร?" ควรใช้ CI หรือ PI? ทำไม?
